In [2]:
import requests  # HTTP 요청을 보내고 응답을 받을 수 있도록 해주는 라이브러리
import xml.etree.ElementTree as ET  # XML 데이터를 파싱(분석)하고 다루기 위한 라이브러리
import pandas as pd  # 데이터프레임을 활용하여 데이터를 효율적으로 처리하기 위한 라이브러리

'''
실습 주제: 서울시 OpenAPI(버스 승하차 통계)에서 XML 데이터를 수집하고 DataFrame으로 변환하기

1) OpenAPI 호출 URL을 조합할 수 있다.
2) requests.get()으로 GET 요청을 보내고 상태코드를 확인할 수 있다.
3) XML 응답을 ElementTree로 파싱하고 <row> 데이터를 추출할 수 있다.
4) 추출한 데이터를 pandas DataFrame으로 만들 수 있다.

# 참고사항
공공데이터의 api를 다루는 사이트는 여러 곳이 있습니다.
그 중에서 과제에 해당하는 사이트: https://data.seoul.go.kr 에 가입 후 인증키를 발급받아야 합니다.

버스 api 사이트: https://data.seoul.go.kr/dataList/OA-12914/S/1/datasetView.do
각각의 사이트에서 인증키를 받아야 하며 첫 번째 이미지의 인증키 신청을 누르고 두 번째의 내용을 작성합니다.
여기서 사용URL이 헷갈릴 수 있는데, 이건 "API 테스트 및 학습용" 정도로 기입해주시면 인증키를 발급받을 수 있고 이 인증키를 
두 개의 사이트에 대해서 발급받고 진행합니다.

- BASE_URL: http://openapi.seoul.go.kr:8088/
- URL 형식:
  {BASE_URL}{API_KEY}/xml/{SERVICE_NAME}/{START_INDEX}/{END_INDEX}/{DATE}

'''

'\n실습 주제: 서울시 OpenAPI(버스 승하차 통계)에서 XML 데이터를 수집하고 DataFrame으로 변환하기\n\n1) OpenAPI 호출 URL을 조합할 수 있다.\n2) requests.get()으로 GET 요청을 보내고 상태코드를 확인할 수 있다.\n3) XML 응답을 ElementTree로 파싱하고 <row> 데이터를 추출할 수 있다.\n4) 추출한 데이터를 pandas DataFrame으로 만들 수 있다.\n\n# 참고사항\n공공데이터의 api를 다루는 사이트는 여러 곳이 있습니다.\n그 중에서 과제에 해당하는 사이트: https://data.seoul.go.kr 에 가입 후 인증키를 발급받아야 합니다.\n\n버스 api 사이트: https://data.seoul.go.kr/dataList/OA-12914/S/1/datasetView.do\n각각의 사이트에서 인증키를 받아야 하며 첫 번째 이미지의 인증키 신청을 누르고 두 번째의 내용을 작성합니다.\n여기서 사용URL이 헷갈릴 수 있는데, 이건 "API 테스트 및 학습용" 정도로 기입해주시면 인증키를 발급받을 수 있고 이 인증키를 \n두 개의 사이트에 대해서 발급받고 진행합니다.\n\n- BASE_URL: http://openapi.seoul.go.kr:8088/\n- URL 형식:\n  {BASE_URL}{API_KEY}/xml/{SERVICE_NAME}/{START_INDEX}/{END_INDEX}/{DATE}\n\n'

In [4]:
# TODO: 본인 인증키로 선언하세요
API_KEY = "6d6d4e474f74726536386474566d5a"

# TODO: 조회할 날짜(YYYYMMDD)로 바꾸세요
DATE = "20260410"

# 서울시 OpenAPI 기본 URL
BASE_URL = "http://openapi.seoul.go.kr:8088/"

# 버스 승하차 통계 API
SERVICE_NAME = "CardBusStatisticsServiceNew"
START_INDEX = 1
END_INDEX = 50

# 문제 1) 아래 URL을 형식에 맞게 완성하세요.
# URL 형식: {BASE_URL}{API_KEY}/xml/{SERVICE_NAME}/{START_INDEX}/{END_INDEX}/{DATE}
url = f"{BASE_URL}{API_KEY}/xml/{SERVICE_NAME}/{START_INDEX}/{END_INDEX}/{DATE}"
print("[1] 요청 URL:")
print(url)

[1] 요청 URL:
http://openapi.seoul.go.kr:8088/6d6d4e474f74726536386474566d5a/xml/CardBusStatisticsServiceNew/1/50/20260410


In [5]:
# 문제 2) requests를 사용해 GET 요청을 보내세요.
response = requests.get(url)

print("\n[2] HTTP 상태 코드:")
print(response.status_code)

# 문제 3) 상태 코드가 200이 아니면 에러를 발생시키세요.
if response.status_code != 200:
    raise RuntimeError("API 호출 실패: 상태코드를 확인하세요.")



[2] HTTP 상태 코드:
200


In [6]:
# 문제 4) XML 파싱: response.content를 사용해 XML을 파싱하세요.
root = ET.fromstring(response.text)


In [7]:
# 문제 5) <row> 태그들을 전부 찾으세요.
rows = root.findall(".//row")
print("\n[3] row 개수:", len(rows))


[3] row 개수: 50


In [8]:
# 문제 6) 각 row에서 필요한 값 4개를 추출해 data 리스트에 딕셔너리 형태로 추가하세요.
# - USE_YMD, RTE_NM, STTN_NM, GTON_TNOPE
data = []
for row in rows:
    use_ymd = row.findtext("USE_YMD", default="")
    route_nm = row.findtext("RTE_NM", default="")
    station_nm = row.findtext("STTN_NM", default="")
    ride = row.findtext("GTON_TNOPE", default="0")

    data.append({
        "날짜": use_ymd,
        "노선번호": route_nm,
        "정류장명": station_nm,
        # 문제 7) ride가 숫자면 int로 변환하고, 아니면 0을 넣으세요.
        "승차인원": int(float(ride)) if ride.replace('.','',1).isdigit() else 0
    })

In [9]:
# 문제 8) data 리스트를 DataFrame으로 변환하세요.
df = pd.DataFrame(data)

print("\n[4] 결과 미리보기:")
print(df.head(10))

print("\n[5] 컬럼 확인:")
print(df.columns.tolist())



[4] 결과 미리보기:
         날짜            노선번호 정류장명  승차인원
0  20260410  100번(하계동~용산구청)        204
1  20260410  100번(하계동~용산구청)        186
2  20260410  100번(하계동~용산구청)        314
3  20260410  100번(하계동~용산구청)         25
4  20260410  100번(하계동~용산구청)        120
5  20260410  100번(하계동~용산구청)        179
6  20260410  100번(하계동~용산구청)         99
7  20260410  100번(하계동~용산구청)        176
8  20260410  100번(하계동~용산구청)         73
9  20260410  100번(하계동~용산구청)         82

[5] 컬럼 확인:
['날짜', '노선번호', '정류장명', '승차인원']
